In [7]:
import pandas as pd
import numpy as np

# Load the dataset
df_ratings = pd.read_csv('ratings.csv')

print("First 5 rows of the dataset:")
print(df_ratings.head())

# Create the user-item matrix
user_item_matrix = df_ratings.pivot_table(index='userId', columns='movieId', values='rating')

print("\nUser-item matrix created.")

FileNotFoundError: [Errno 2] No such file or directory: 'ratings.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
import pandas as pd
import numpy as np
import os
import zipfile
import requests

# Define the URL for the MovieLens dataset (latest small version)
movielens_url = "http://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
zip_file_path = "/tmp/ml-latest-small.zip"
extract_path = "/tmp/ml-latest-small"
ratings_csv_path = os.path.join(extract_path, "ratings.csv")

# Check if ratings.csv already exists
if not os.path.exists(ratings_csv_path):
    print(f"Downloading MovieLens dataset from {movielens_url}...")
    response = requests.get(movielens_url)
    response.raise_for_status() # Raise an exception for bad status codes

    with open(zip_file_path, "wb") as f:
        f.write(response.content)
    print(f"Dataset downloaded to {zip_file_path}.")

    print(f"Extracting {zip_file_path}...")
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"Dataset extracted to {extract_path}.")
else:
    print(f"'ratings.csv' already found at {ratings_csv_path}, skipping download and extraction.")

# Load the dataset
df_ratings = pd.read_csv(ratings_csv_path)

print("\nFirst 5 rows of the dataset:")
print(df_ratings.head())

# Create the user-item matrix
user_item_matrix = df_ratings.pivot_table(index='userId', columns='movieId', values='rating')

print("\nUser-item matrix created.")
print("\nFirst 5 rows and columns of the user-item matrix:")
print(user_item_matrix.iloc[:5, :5])

Dataset downloaded to /tmp/ml-latest-small.zip.
Extracting /tmp/ml-latest-small.zip...
Dataset extracted to /tmp/ml-latest-small.


FileNotFoundError: [Errno 2] No such file or directory: '/tmp/ml-latest-small/ratings.csv'

In [4]:
import pandas as pd
import numpy as np
import os
import zipfile
import requests

# Define the URL for the MovieLens dataset (latest small version)
movielens_url = "http://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
zip_file_path = "/tmp/ml-latest-small.zip"
extract_path = "/tmp/ml-latest-small"

# Adjust ratings_csv_path to account for the nested directory structure inside the zip file
ratings_csv_path = os.path.join(extract_path, "ml-latest-small", "ratings.csv")

# Check if ratings.csv already exists
if not os.path.exists(ratings_csv_path):
    print(f"Downloading MovieLens dataset from {movielens_url}...")
    response = requests.get(movielens_url)
    response.raise_for_status() # Raise an exception for bad status codes

    with open(zip_file_path, "wb") as f:
        f.write(response.content)
    print(f"Dataset downloaded to {zip_file_path}.")

    print(f"Extracting {zip_file_path}...")
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"Dataset extracted to {extract_path}.")
else:
    print(f"'ratings.csv' already found at {ratings_csv_path}, skipping download and extraction.")

# Load the dataset
df_ratings = pd.read_csv(ratings_csv_path)

print("\nFirst 5 rows of the dataset:")
print(df_ratings.head())

# Create the user-item matrix
user_item_matrix = df_ratings.pivot_table(index='userId', columns='movieId', values='rating')

print("\nUser-item matrix created.")
print("\nFirst 5 rows and columns of the user-item matrix:")
print(user_item_matrix.iloc[:5, :5])

'ratings.csv' already found at /tmp/ml-latest-small/ml-latest-small/ratings.csv, skipping download and extraction.

First 5 rows of the dataset:
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931

User-item matrix created.

First 5 rows and columns of the user-item matrix:
movieId    1   2    3   4   5
userId                       
1        4.0 NaN  4.0 NaN NaN
2        NaN NaN  NaN NaN NaN
3        NaN NaN  NaN NaN NaN
4        NaN NaN  NaN NaN NaN
5        4.0 NaN  NaN NaN NaN


In [5]:
import numpy as np

# 1. Replace missing values (NaN) with 0
user_item_matrix_filled = user_item_matrix.fillna(0)

print("User-item matrix after filling NaNs with 0 (first 5 rows and columns):")
print(user_item_matrix_filled.iloc[:5, :5])

# 2. Convert the preprocessed user_item_matrix into a NumPy array
R = user_item_matrix_filled.values

# 3. Apply Singular Value Decomposition (SVD)
U, s, Vh = np.linalg.svd(R, full_matrices=False)

print(f"\nShape of U: {U.shape}")
print(f"Shape of s: {s.shape}")
print(f"Shape of Vh: {Vh.shape}")

# 4. Reconstruct the user-latent feature matrix by taking the first 'k' singular values and corresponding vectors
k = 50  # Number of latent features

# Truncate U to k columns
U_k = U[:, :k]

# Create a diagonal matrix from the first k singular values
s_k = np.diag(s[:k])

# Truncate Vh to k rows (k latent features)
Vh_k = Vh[:k, :]

# User-latent feature matrix (U * s_k)
user_latent_features = np.dot(U_k, s_k)

print(f"\nShape of user_latent_features (U * s_k): {user_latent_features.shape}")

# Convert user_latent_features to a DataFrame for better readability and future use
user_latent_features_df = pd.DataFrame(user_latent_features,
                                       index=user_item_matrix_filled.index,
                                       columns=[f'latent_feature_{i+1}' for i in range(k)])

print("\nFirst 5 rows of the user-latent feature matrix:")
print(user_latent_features_df.head())

User-item matrix after filling NaNs with 0 (first 5 rows and columns):
movieId    1    2    3    4    5
userId                          
1        4.0  0.0  4.0  0.0  0.0
2        0.0  0.0  0.0  0.0  0.0
3        0.0  0.0  0.0  0.0  0.0
4        0.0  0.0  0.0  0.0  0.0
5        4.0  0.0  0.0  0.0  0.0

Shape of U: (610, 610)
Shape of s: (610,)
Shape of Vh: (610, 9724)

Shape of user_latent_features (U * s_k): (610, 50)

First 5 rows of the user-latent feature matrix:
        latent_feature_1  latent_feature_2  latent_feature_3  \
userId                                                         
1             -29.689244         14.261252         -2.083057   
2              -3.135065         -4.101610         -0.845547   
3              -0.723193          0.478339          0.327857   
4             -17.294912         12.540074          6.530639   
5              -6.097046          5.338321         -9.279336   

        latent_feature_4  latent_feature_5  latent_feature_6  \
userId          

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

# 1. Merge df_ratings with user_latent_features_df
df_merged = pd.merge(df_ratings, user_latent_features_df, on='userId', how='left')

print("First 5 rows of the merged DataFrame:")
print(df_merged.head())

First 5 rows of the merged DataFrame:
   userId  movieId  rating  timestamp  latent_feature_1  latent_feature_2  \
0       1        1     4.0  964982703        -29.689244         14.261252   
1       1        3     4.0  964981247        -29.689244         14.261252   
2       1        6     4.0  964982224        -29.689244         14.261252   
3       1       47     5.0  964983815        -29.689244         14.261252   
4       1       50     5.0  964982931        -29.689244         14.261252   

   latent_feature_3  latent_feature_4  latent_feature_5  latent_feature_6  \
0         -2.083057          0.141344        -14.241235          7.736142   
1         -2.083057          0.141344        -14.241235          7.736142   
2         -2.083057          0.141344        -14.241235          7.736142   
3         -2.083057          0.141344        -14.241235          7.736142   
4         -2.083057          0.141344        -14.241235          7.736142   

   ...  latent_feature_41  latent_fe

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor

# 2. Define features (X) and target (y)
# Features include userId, movieId, and all latent_feature_X columns
feature_columns = ['userId', 'movieId'] + [col for col in df_merged.columns if 'latent_feature_' in str(col)]
X = df_merged[feature_columns]
y = df_merged['rating']

# 3. Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

# 4. Initialize and train a GradientBoostingRegressor model
print("\nTraining Gradient Boosting Regressor...")
gb_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_model.fit(X_train, y_train)
print("Gradient Boosting Regressor training complete.")

# 5. Predict ratings on the test data
gb_predictions = gb_model.predict(X_test)

print("\nFirst 10 Gradient Boosting predictions:")
print(gb_predictions[:10])

Shape of X_train: (80668, 52)
Shape of X_test: (20168, 52)
Shape of y_train: (80668,)
Shape of y_test: (20168,)

Training Gradient Boosting Regressor...
Gradient Boosting Regressor training complete.

First 10 Gradient Boosting predictions:
[3.56802715 3.12383391 2.59817266 3.59213083 3.51622819 3.6125856
 3.67743518 3.63398229 3.38787086 3.64694738]


In [9]:
from sklearn.preprocessing import StandardScaler

# 6. For the Neural Network, normalize or scale the feature data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Shape of X_train_scaled: {X_train_scaled.shape}")
print(f"Shape of X_test_scaled: {X_test_scaled.shape}")

print("First 5 rows of X_train_scaled (first 5 features):")
print(X_train_scaled[:5, :5])

Shape of X_train_scaled: (80668, 52)
Shape of X_test_scaled: (20168, 52)
First 5 rows of X_train_scaled (first 5 features):
[[ 0.99830454 -0.34134737  0.33115532 -1.16228639 -0.279353  ]
 [-0.00295088  1.46267314  0.82322858 -0.47715261 -0.40088194]
 [-1.47474164 -0.48856152 -0.06185815  1.63336905 -0.20817359]
 [ 1.55090999 -0.51636113 -1.24596842 -2.69323702  0.95673294]
 [ 0.74115151 -0.48028917  0.39259129  0.48180729  0.52339711]]


In [10]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 7. Design a simple feedforward Neural Network
# The input layer should match the number of features in X_train_scaled
input_shape = X_train_scaled.shape[1]

model_nn = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(input_shape,)),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)  # Output layer for regression (single neuron, linear activation by default)
])

# 8. Compile the Neural Network
model_nn.compile(optimizer='adam', loss='mean_squared_error', metrics=['mean_absolute_error', 'mean_squared_error'])

print("Neural Network model designed and compiled.")
model_nn.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Neural Network model designed and compiled.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         6,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,153 (67.00 KB)

 Trainable params: 17,153 (67.00 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 9. Train the Neural Network model
print("\nTraining Neural Network...")
history = model_nn.fit(
    X_train_scaled, y_train,
    epochs=10,  # You can adjust the number of epochs
    batch_size=64,
    validation_split=0.2, # Use 20% of training data for validation
    verbose=1 # Show training progress
)
print("Neural Network training complete.")

# 10. Predict ratings on the scaled test data
nn_predictions = model_nn.predict(X_test_scaled)

print("\nFirst 10 Neural Network predictions:")
print(nn_predictions[:10].flatten())


Training Neural Network...
Epoch 1/10
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 2.1177 - mean_absolute_error: 1.1062 - mean_squared_error: 2.1177 - val_loss: 1.0225 - val_mean_absolute_error: 0.8092 - val_mean_squared_error: 1.0225
Epoch 2/10
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 1.0263 - mean_absolute_error: 0.7966 - mean_squared_error: 1.0263 - val_loss: 1.0299 - val_mean_absolute_error: 0.8261 - val_mean_squared_error: 1.0299
Epoch 3/10
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.9804 - mean_absolute_error: 0.7779 - mean_squared_error: 0.9804 - val_loss: 0.9982 - val_mean_absolute_error: 0.8049 - val_mean_squared_error: 0.9982
Epoch 4/10
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.9525 - mean_absolute_error: 0.7653 - mean_squared_error: 0.9525 - val_loss: 0.9463 - val_mean_absolute_error: 0.7656 - val_mean_squared_error: 0.9463
Epoch 5/10
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.9522 - mean_absolute_error: 0.7643 - mean_squared_e

In [12]:
from sklearn.metrics import mean_squared_error, precision_score, recall_score
import numpy as np

# 1. Evaluate Gradient Boosting Regressor
rmse_gb = np.sqrt(mean_squared_error(y_test, gb_predictions))

# For Precision and Recall, we need to binarize ratings. Let's assume a rating >= 3.5 is a 'positive' recommendation.
threshold = 3.5
y_test_binary = (y_test >= threshold).astype(int)
gb_predictions_binary = (gb_predictions >= threshold).astype(int)

precision_gb = precision_score(y_test_binary, gb_predictions_binary)
recall_gb = recall_score(y_test_binary, gb_predictions_binary)

print("\n--- Gradient Boosting Regressor Performance ---")
print(f"RMSE: {rmse_gb:.4f}")
print(f"Precision (threshold={threshold}): {precision_gb:.4f}")
print(f"Recall (threshold={threshold}): {recall_gb:.4f}")

# 2. Evaluate Neural Network Model
# Flatten nn_predictions which is a 2D array (n_samples, 1) to 1D
nn_predictions_flat = nn_predictions.flatten()
rmse_nn = np.sqrt(mean_squared_error(y_test, nn_predictions_flat))

nn_predictions_binary = (nn_predictions_flat >= threshold).astype(int)

precision_nn = precision_score(y_test_binary, nn_predictions_binary)
recall_nn = recall_score(y_test_binary, nn_predictions_binary)

print("\n--- Neural Network Model Performance ---")
print(f"RMSE: {rmse_nn:.4f}")
print(f"Precision (threshold={threshold}): {precision_nn:.4f}")
print(f"Recall (threshold={threshold}): {recall_nn:.4f}")



--- Gradient Boosting Regressor Performance ---
RMSE: 0.9510
Precision (threshold=3.5): 0.7412
Recall (threshold=3.5): 0.6712

--- Neural Network Model Performance ---
RMSE: 0.9587
Precision (threshold=3.5): 0.7442
Recall (threshold=3.5): 0.6376


In [13]:
import numpy as np

# Ensure nn_predictions is flattened for averaging
nn_predictions_flat = nn_predictions.flatten()

# 1. Average the predictions from the Gradient Boosting model and the Neural Network model
hybrid_predictions = (gb_predictions + nn_predictions_flat) / 2

# 2. Print the first 10 values of hybrid_predictions
print("\nFirst 10 Hybrid Predictions:")
print(hybrid_predictions[:10])


First 10 Hybrid Predictions:
[3.52853865 3.13479973 2.607283   3.57112214 3.4840995  3.70569172
 3.64017821 3.55632144 3.38402549 3.6194148 ]
